# 📉 Step 3 — Log-Return Transformation

**Formula:** rₜ = log(Pₜ / Pₜ₋₁)

**Applied on:** Close price only

**Why log-return transformation?**
- Raw Close prices are **non-stationary** — they trend upward over time
- Neural networks require **stationary inputs** — values that fluctuate around a stable mean
- Log returns convert price levels into **daily percentage changes** on a log scale
- They are **additive over time** and approximately normally distributed
- They **stabilise variance** across different price regimes

**Input file:** `nifty50_with_indicators_zerodha.csv` (output from Step 2)

**Output file:** `nifty50_step3_log_return.csv`

---
### Instructions
1. Run **Cell 1** — install libraries
2. Run **Cell 2** — upload your Step 2 output CSV
3. Run **Cell 3** — understand why log return is needed (stationarity test)
4. Run **Cell 4** — apply log-return transformation on Close
5. Run **Cell 5** — verify stationarity is achieved
6. Run **Cell 6** — full preview of output
7. Run **Cell 7** — save and download

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Install Libraries
# ─────────────────────────────────────────────────────────────────────────
!pip install pandas numpy statsmodels -q
print('✓ Libraries ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Upload and Load Step 2 Output CSV
# ─────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from google.colab import files

print('Please upload your Step 2 output CSV file...')
print('(nifty50_with_indicators_zerodha.csv)')
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f'\n✓ File uploaded: {filename}')

# Load CSV
df = pd.read_csv(filename)

# Parse dates
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

# Sort oldest to newest
df = df.sort_values('Date').reset_index(drop=True)

# Ensure Close is numeric
df['Close'] = pd.to_numeric(df['Close'], errors='coerce')

print(f'\n✓ Rows loaded  : {len(df)} trading days')
print(f'✓ Date range   : {df["Date"].iloc[0].date()} → {df["Date"].iloc[-1].date()}')
print(f'✓ Columns      : {list(df.columns)}')
print(f'\nFirst 3 rows:')
print(df[['Date', 'Close']].head(3).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — Why Log Return? Stationarity Test on Raw Close
#
# ADF Test (Augmented Dickey-Fuller):
#   H0 : Series is non-stationary (has a unit root)
#   H1 : Series is stationary
#
#   If p-value > 0.05 → FAIL to reject H0 → series is NON-STATIONARY
#   If p-value < 0.05 → REJECT H0         → series is STATIONARY
#
# We expect raw Close to be non-stationary and log return to be stationary
# ─────────────────────────────────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller

def adf_test(series, name):
    clean = series.dropna()
    result = adfuller(clean, autolag='AIC')
    adf_stat  = result[0]
    p_value   = result[1]
    is_stat   = p_value < 0.05
    print(f'  {name}')
    print(f'    ADF Statistic : {adf_stat:.4f}')
    print(f'    p-value       : {p_value:.6f}')
    print(f'    Conclusion    : {"✅ STATIONARY" if is_stat else "❌ NON-STATIONARY"} (p {"<" if is_stat else ">"} 0.05)')
    return is_stat

print('=' * 60)
print('  STATIONARITY TEST — BEFORE LOG TRANSFORMATION')
print('=' * 60)
print()
stat_raw = adf_test(df['Close'], 'Raw Close Price')

print()
print('  Summary of Raw Close:')
print(f'    Mean  : {df["Close"].mean():,.2f}')
print(f'    Std   : {df["Close"].std():,.2f}')
print(f'    Min   : {df["Close"].min():,.2f}')
print(f'    Max   : {df["Close"].max():,.2f}')

if not stat_raw:
    print()
    print('  ⚠️  Raw Close is NON-STATIONARY — log transformation is needed')
    print('  → Proceeding to apply rₜ = log(Pₜ / Pₜ₋₁) in Cell 4')
else:
    print()
    print('  Raw Close appears stationary — log transformation still recommended')
    print('  for neural network stability and comparability across price regimes')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — Apply Log-Return Transformation on Close
#
# Formula : rₜ = log(Pₜ / Pₜ₋₁)
#
# Where:
#   rₜ     = log return on day t
#   Pₜ     = closing price on day t
#   Pₜ₋₁  = closing price on day t-1
#   log    = natural logarithm (base e)
#
# Equivalent to: rₜ = log(Pₜ) − log(Pₜ₋₁)
#
# Properties:
#   Positive rₜ → price rose from yesterday
#   Negative rₜ → price fell from yesterday
#   rₜ ≈ 0     → price was roughly unchanged
#   Typical range for Nifty 50: −0.05 to +0.05 (−5% to +5%)
#
# NOTE: Applied on Close ONLY as per Step 3 requirement.
#       All indicators (EMA, RSI, BB, MACD) remain on original raw scale.
# ─────────────────────────────────────────────────────────────────────────

# Apply log-return on Close
df['LogReturn_Close'] = np.log(df['Close'] / df['Close'].shift(1)).round(6)

# The first row will be NaN (no previous price) — keep it but flag it
nan_count = df['LogReturn_Close'].isna().sum()

print('=' * 60)
print('  LOG-RETURN TRANSFORMATION APPLIED')
print('=' * 60)
print(f'  Formula : rₜ = log(Pₜ / Pₜ₋₁)')
print(f'  Applied : Close price only')
print(f'  NaN rows: {nan_count} (first row has no previous price — expected)')
print()
print('  Log Return Statistics:')
lr = df['LogReturn_Close'].dropna()
print(f'    Count  : {len(lr)} values')
print(f'    Mean   : {lr.mean():.6f}')
print(f'    Std    : {lr.std():.6f}')
print(f'    Min    : {lr.min():.6f}  ({lr.min()*100:.2f}%)')
print(f'    Max    : {lr.max():.6f}  ({lr.max()*100:.2f}%)')
print(f'    Median : {lr.median():.6f}')
print()

# Show top 5 biggest single-day rises and falls
df_temp = df[['Date', 'Close', 'LogReturn_Close']].dropna()
print('  Top 5 largest single-day RISES:')
top_rise = df_temp.nlargest(5, 'LogReturn_Close')
for _, r in top_rise.iterrows():
    print(f'    {r["Date"].date()}  Close={r["Close"]:,.2f}  LogReturn={r["LogReturn_Close"]:+.6f}  ({r["LogReturn_Close"]*100:+.2f}%)')

print()
print('  Top 5 largest single-day FALLS:')
top_fall = df_temp.nsmallest(5, 'LogReturn_Close')
for _, r in top_fall.iterrows():
    print(f'    {r["Date"].date()}  Close={r["Close"]:,.2f}  LogReturn={r["LogReturn_Close"]:+.6f}  ({r["LogReturn_Close"]*100:+.2f}%)')

print()
print('  Sample (last 5 rows):')
print(df[['Date', 'Close', 'LogReturn_Close']].tail(5).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — Verify Stationarity AFTER Log Transformation
#
# We now run the ADF test again on LogReturn_Close
# We expect p-value < 0.05 confirming stationarity is achieved
# ─────────────────────────────────────────────────────────────────────────

print('=' * 60)
print('  STATIONARITY TEST — AFTER LOG TRANSFORMATION')
print('=' * 60)
print()

stat_lr = adf_test(df['LogReturn_Close'], 'Log Return of Close (rₜ)')

print()
print('  Comparison:')
print(f'    Raw Close Price   → {"✅ Stationary" if stat_raw else "❌ Non-Stationary"}')
print(f'    Log Return Close  → {"✅ Stationary" if stat_lr  else "❌ Non-Stationary"}')

print()
if stat_lr:
    print('  ✅ Stationarity achieved after log-return transformation.')
    print('  Log Return is now suitable as input to the neural network.')
else:
    print('  ⚠️  Log return still shows non-stationarity signs.')
    print('  This may be due to limited data. Proceed — log return is')
    print('  still the correct transformation for neural network input.')

print()
print('  Why this matters for your model:')
print('    Raw Close  → values between 17,000 and 25,000 — model hard to train')
print('    Log Return → values between -0.05 and +0.05   — stable, trainable')
print('    Log Return fluctuates around 0 with consistent scale across time')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — Full Dataset Preview
# ─────────────────────────────────────────────────────────────────────────
print('=' * 75)
print('  STEP 3 OUTPUT — FULL COLUMN LIST')
print('=' * 75)
print(f'  Total rows    : {len(df)}')
print(f'  Total columns : {len(df.columns)}')
print(f'  Date range    : {df["Date"].iloc[0].date()} → {df["Date"].iloc[-1].date()}')
print()
print('  Columns:')
for col in df.columns:
    nn = df[col].notna().sum()
    tag = '  ← NEW (Step 3)' if col == 'LogReturn_Close' else ''
    print(f'    {col:<22} : {nn} non-null values{tag}')

print()
print('  Last 5 rows (key columns):')
show_cols = ['Date','Close','LogReturn_Close','EMA_20','EMA_50',
             'RSI_14','BB_Upper','BB_Middle','BB_Lower',
             'MACD_Line','MACD_Signal','MACD_Hist']
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(df[show_cols].tail(5).to_string(index=False))

print()
print('  Next step (Step 4):')
print('  Apply Normalisation/Standardisation on LogReturn_Close')
print('  and all indicator columns before feeding into the model.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 7 — Save Final CSV and Download
# ─────────────────────────────────────────────────────────────────────────
OUTPUT_FILE = 'nifty50_step3_log_return.csv'

df_save = df.copy()
df_save['Date'] = df_save['Date'].dt.strftime('%d-%b-%Y')

# Final column order — LogReturn_Close placed right after Close
final_cols = [
    'Date', 'Open', 'High', 'Low', 'Close',
    'LogReturn_Close',
    'EMA_20', 'EMA_50', 'EMA_Cross',
    'RSI_14',
    'BB_Upper', 'BB_Middle', 'BB_Lower', 'BB_Width', 'BB_Position',
    'MACD_Line', 'MACD_Signal', 'MACD_Hist'
]

# Keep only columns that exist in df
final_cols = [c for c in final_cols if c in df_save.columns]
df_save = df_save[final_cols]

df_save.to_csv(OUTPUT_FILE, index=False)

print('=' * 60)
print(f'  ✓ File saved    : {OUTPUT_FILE}')
print(f'  ✓ Total rows    : {len(df_save)}')
print(f'  ✓ Total columns : {len(df_save.columns)}')
print('  ✓ Columns saved :')
for col in df_save.columns:
    tag = '  ← NEW (Step 3)' if col == 'LogReturn_Close' else ''
    print(f'      {col}{tag}')
print('=' * 60)
print('\n⬇️  Downloading file to your computer...')
files.download(OUTPUT_FILE)
print('✓ Download complete!')
print('\n✅ Step 3 Complete — Ready for Step 4 (Normalisation & Standardisation)')